In [36]:
from fileformer import utils, Decoder, Encoder, ByteLevelTokenizer

In [37]:
import torch
from flax import nnx
from torchinfo import summary

In [38]:
decoder = Decoder(258, 128, 1, 1, 256, 0.0, 64, 'gelu')

In [39]:
summary(decoder)

Layer (type:depth-idx)                        Param #
Decoder                                       --
├─Embedding: 1-1                              33,024
├─RotaryPositionalEmbeddings: 1-2             --
├─ModuleList: 1-3                             --
│    └─DecoderLayer: 2-1                      --
│    │    └─MultiHeadLatentAttention: 3-1     82,624
│    │    └─MLP: 3-2                          65,920
│    │    └─LayerNorm: 3-3                    256
│    │    └─LayerNorm: 3-4                    256
│    │    └─Dropout: 3-5                      --
├─Dropout: 1-4                                --
├─Linear: 1-5                                 33,282
Total params: 215,362
Trainable params: 215,362
Non-trainable params: 0

In [40]:
from fileformer import ENWIK8Dataset
from fileformer.tokenizer import ByteLevelTokenizer
import torch
from torch import Tensor
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [41]:
decoder.eval()

Decoder(
  (chunk_emb): Embedding(258, 128)
  (pe): RotaryPositionalEmbeddings()
  (layers): ModuleList(
    (0): DecoderLayer(
      (self_attention): MultiHeadLatentAttention(
        (latent_proj): Linear(in_features=128, out_features=64, bias=True)
        (latent_proj_back): Linear(in_features=64, out_features=128, bias=True)
        (q_proj): Linear(in_features=128, out_features=128, bias=True)
        (k_proj): Linear(in_features=128, out_features=128, bias=True)
        (v_proj): Linear(in_features=128, out_features=128, bias=True)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (mlp): MLP(
        (activation): GELU(approximate='none')
        (mlp): Sequential(
          (0): Linear(in_features=128, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=128, bias=True)
          (3): Dropout(p=0.0, inplace=False)
        )
 

In [42]:
x = torch.tensor([[148, 67]], dtype=torch.long)
mask = torch.zeros_like(x).to(torch.bool)

In [43]:
x.shape

torch.Size([1, 2])

In [44]:
mask

tensor([[False, False]])

In [45]:
out = decoder(x, mask)

In [46]:
pred = torch.argmax(out, dim=-1)[:, 1:]

In [47]:
v = torch.cat([x, pred], dim=1)

In [48]:
v

tensor([[148,  67,  63]])

In [49]:
def generate(input, step):
    for _ in range(step):
        mask = torch.zeros_like(input).to(torch.bool)
        out = decoder(input, mask)
        pred = torch.argmax(out, dim=-1)[:, -1:]
        input = torch.cat([input, pred], dim=1)
    return input
        
        

In [50]:
gen = generate(x, 25)

In [51]:
token = ByteLevelTokenizer()

In [52]:
token.decode(gen.tolist()[0])

'92413dc0860a86c332f2ae575639e286cdfbab07a0b66f724c<mask>d6'

In [ ]:
gen.shape